In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("clean_data.csv")

X = df['clean_text']
y = df['incident_category']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training size:", len(X_train))
print("Testing size:", len(X_test))

Training size: 4000
Testing size: 1000


In [7]:
cv = CountVectorizer()
X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)

print("Count Vectorizer shape:", X_train_cv.shape)
print("Sparsity:", round(1 - X_train_cv.nnz / (X_train_cv.shape[0] * X_train_cv.shape[1]), 4))

Count Vectorizer shape: (4000, 245)
Sparsity: 0.9653


In [9]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF shape:", X_train_tfidf.shape)
print("Sparsity:", round(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]), 4))

TF-IDF shape: (4000, 245)
Sparsity: 0.9653


In [11]:
model_cv = LogisticRegression(max_iter=1000)
model_cv.fit(X_train_cv, y_train)
acc_cv = accuracy_score(y_test, model_cv.predict(X_test_cv))

model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train_tfidf, y_train)
acc_tfidf = accuracy_score(y_test, model_tfidf.predict(X_test_tfidf))

print("Count Vectorizer Accuracy:", round(acc_cv, 4))
print("TF-IDF Accuracy:", round(acc_tfidf, 4))

Count Vectorizer Accuracy: 1.0
TF-IDF Accuracy: 1.0


In [13]:
comparison = pd.DataFrame({
    'Method': ['Count Vectorizer', 'TF-IDF'],
    'Feature Dimensions': [X_train_cv.shape[1], X_train_tfidf.shape[1]],
    'Sparsity': [
        round(1 - X_train_cv.nnz / (X_train_cv.shape[0] * X_train_cv.shape[1]), 4),
        round(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]), 4)
    ],
    'Accuracy': [round(acc_cv, 4), round(acc_tfidf, 4)]
})

comparison

,Method,Feature Dimensions,Sparsity,Accuracy
0,Count Vectorizer,245,0.9653,1.0
1,TF-IDF,245,0.9653,1.0


### Observations
**Feature Dimensions - both 245**

Both methods found exactly 245 unique words in the training data. This makes sense because both use the same vocabulary but score the words differently.

**Sparsity - both 0.9653**

96.5% of the matrix is zeros. This means each ticket only contains a small fraction of the total 245 words, which is expected given how short and templated the descriptions are.

**Accuracy - both 1.0**

Both methods achieved 100% accuracy using Logistic Regression. This sounds impressive but is directly linked to the templated nature of the dataset. Each category has such distinct and consistent keywords that even a basic model separates them perfectly. On real messy incident data, TF-IDF would likely pull ahead of Count Vectorizer because it downweights template words like 'report' and 'facility' that appear everywhere, giving more weight to the words that actually distinguish categories.

In [18]:
import scipy.sparse as sp
import numpy as np

sp.save_npz("X_train_cv.npz", X_train_cv)
sp.save_npz("X_test_cv.npz", X_test_cv)
sp.save_npz("X_train_tfidf.npz", X_train_tfidf)
sp.save_npz("X_test_tfidf.npz", X_test_tfidf)

np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)

print("Saved.")

Saved.
